## Notas Caio

#### Etapa ingestão (raw)
- Mencionar passo-a-passo do bucket s3 + external data c/ databricks + gravar secrets (ARN) de maneira segura (sem vazamento)
- Mencionar que tambem é necessario criar um schema e volume para associar o bucket ao databricks
- Pasta renomeada de `ingestion/` para `raw/` para tornar a progressão das camadas legível direto na estrutura de pastas: `raw/ → bronze/ → silver/ → gold/`

#### Etapa de metadata analysis antes de bronze
- Comentar a importancia de visualizar schema (metadata), entender antes de definir o schema enforcement
- Com os resultados, foi possivel definir Upscating para cada coluna

#### Etapa bronze
- Comentar a questao de DLT e por isso utilizaremos o script como base para um job
- Ingerimos com Upscating baseado na analise de metadados para compatibilidade
- **Contrato de metadados (`table_metadata.py`):** dataclasses `ColumnMeta` / `TableMeta` (frozen=True) centralizam descrições e tipos de cast. `_BRONZE_CASTS` dict separado das descrições — função `_col()` injeta `cast_to` via `.get()` mantendo o catálogo de colunas limpo. Descrições de tabela variam por camada (constantes `BRONZE/SILVER/GOLD_TABLE_COMMENT`); descrições de colunas são universais e reutilizadas. Aplicadas via `COMMENT ON TABLE` + `ALTER TABLE ... ALTER COLUMN COMMENT` no Unity Catalog após o `saveAsTable()`. Função `_apply_table_metadata()` ignora silenciosamente colunas ausentes na tabela — permite reutilizar o mesmo `TAXI_META` em Silver e Gold sem erro.

#### Etapa silver
- Explicar decisao de manter o schema full como Single Source of Truth
- Nao faremos downcasting pelo volume de dados da base para o desafio, porem em um cenario produtivo com BigData justificaria. No entanto, seguiremos com estrategia de particionamento (year/month)
- **EDA Silver (`02_silver_eda.ipynb`) definiu as seguintes regras de cleansing:**
  - **Remove** duplicatas por chave composta `(VendorID, pickup, dropoff, PULocationID, DOLocationID)` — re-transmissão de store-and-forward (~125k linhas, 0,77%)
  - **Remove** viagens no tempo (`dropoff < pickup`) — impossibilidade física (~795 linhas, 0,005%)
  - **Remove** datas fora do range Jan–Mai 2023 — corrupção na origem (~104 linhas)
  - **Mantém** `total_amount < 0` — maioria é `payment_type=4` (Dispute), representando chargebacks legítimos. Decisão de filtrar é do consumidor (Gold)
  - **Mantém** NULLs sistêmicos (428k linhas com 5 colunas nulas simultaneamente) — falha do fornecedor em campos secundários; registro ainda tem âncora temporal e valor financeiro válidos
  - **Mantém** `passenger_count == 0` e `trip_distance == 0` — decisões de negócio para Gold

#### Etapa gold
- Manter apenas colunas necessarias para analise conforme requisito do case
- Enriquecer colunas de código numérico (`VendorID`, `RatecodeID`, `payment_type`) com descrições legíveis via `CASE WHEN` — camada de consumo deve ser auto-explicativa para o analista
- **Não implementar tabelas dimensão:** para o escopo delimitado do case e baixo volume de dados, regras de negócio via `CASE WHEN` são suficientes. Em um cenário produtivo e visando escala, a modelagem dimensional (Kimball/star schema) seria o caminho correto — dims de fornecedor, tarifa e pagamento em Silver, com fact table em Gold
- `PULocationID` / `DOLocationID`: mantidos como IDs numéricos — arquivo de lookup de zonas TLC não está no escopo do dicionário recebido e não é exigido pelas perguntas do case

#### *Apresentaçao
- [Ingestao] Comentar como deixamos modularizado, pode escalar para meses ou anos distintos alterando em settings.py
- [Geral] Comentar sobre a abordagem de logging com decorator, para manter observabilidade padronizada entre modulos garantindo DRY
- [Delta live tables] Comentar que em produçao, utilizariamos o framework Delta Live Tables (DLT) integrado ao Auto Loader, garantindo ingestão contínua em streaming, evolução de schema nativa (rescued data) e monitoramento de qualidade via Data Expectations
- [Delta Tables - Overwrite] Poucos dados/estaticos optei pelo modo overwrite para garantir a idempotência. Cenário de produção a abordagem correta seria o .mode("append") aliado ao particionamento físico da tabela por ano e mes, utilizando a configuração de Dynamic Partition Overwrite
- [Funçoes sem return] Persistência em cada camada. Como as funções geram side-effects diretos na infraestrutura (gravando fisicamente em Buckets e Delta Tables), o estado é salvo no disco. Isso elimina a necessidade de retornar objetos (DataFrames) em memória (in-memory) para o orquestrador, garantindo resiliência

## Notas IA

### Fundação de Storage e Ingestão

**Fase 0 — Storage (Decoupled Storage and Compute)**
O dado bruto (~16M registros, Jan–Mai 2023) é armazenado em Object Storage (AWS S3, mapeado como Databricks External Volume via `RAW_VOLUME_PATH`). Isso garante armazenamento infinito e barato, completamente desacoplado do custo de processamento — pilar do padrão Data Lakehouse.

**Fase 1 — Ingestão ELT (As-Is / Imutável)**
Adotamos ELT, não ETL. A regra da camada Raw é: o dado aterra no bucket exatamente como nasceu na origem, sem nenhuma alteração. A evidência de que isso foi a decisão correta são os conflitos de schema entre meses (ex: `passenger_count` como `double` em Janeiro e `bigint` em Fevereiro) descobertos na EDA. Se o script de extração tivesse tentado "consertar" durante o download, o problema teria sido mascarado e o histórico da origem perdido.

**Fase 2 — EDA de Schema (01_metadata_analysis.ipynb)**
Antes de qualquer transformação, realizamos uma análise de metadata cruzando os schemas dos 5 arquivos Parquet mês a mês. Esse passo identificou os conflitos de tipo que fundamentaram o contrato de upcasting da Bronze.

**Fase 3 — Bronze: Schema Enforcement + Contrato de Metadados**
Ingestão dos Parquets para Delta Table com upcasting seletivo e aplicação de descrições de tabela e colunas via Unity Catalog.

**Fase 4 — EDA Silver (02_silver_eda.ipynb)**
Análise de qualidade sobre a base Bronze para definir as regras de cleansing da Silver: perfil de NULLs, correlação com VendorID, anomalias estruturais, duplicatas e decisão sobre o que é regra estrutural vs. regra de negócio.

**Fase 5 — Silver: Limpeza Estrutural (em implementação)**
Delta Table com dedup, remoção de impossibilidades físicas e particionamento por `year`/`month`. Todas as 19 colunas preservadas como Single Source of Truth.

**Fase 6 — Gold: Projeção Analítica (a implementar)**
Camada de consumo com apenas as colunas necessárias para responder às perguntas de negócio do case.

### Decisões Core de Design e Arquitetura

**1. Schema Enforcement na Camada Bronze**
*Problema:* Os Parquets da TLC possuem conflitos severos de tipo entre meses (`passenger_count` como `double` vs `bigint`; inconsistências de capitalização). `mergeSchema` falhou no Databricks Serverless devido a conflitos estruturais irrecuperáveis; configurações de baixo nível do motor (`spark.sql.caseSensitive`) são bloqueadas pelo ambiente.
*Decisão:* Contrato de dados estrito via `StructType`. Colunas com conflito tipadas para o maior tamanho físico encontrado (`LongType`) — upcasting seletivo, não universal, para manter rastreabilidade de quais colunas tinham problema real. O contrato de quais colunas sofrem upcasting vive em `_BRONZE_CASTS` no `table_metadata.py`, separado das descrições de negócio.

**2. EDA 100% PySpark Nativo**
*Problema:* Profiling de 16M de linhas sem estourar memória. `dbutils.data.summarize()` não é suportado no Serverless (rodava no Driver).
*Decisão:* Toda análise exploratória via funções distribuídas do PySpark (`F.sum`, `F.when`, `.groupBy()`). Roda nos workers, é portável para qualquer cluster e infinitamente escalável.

**3. Contrato de Metadados Centralizado (`table_metadata.py`)**
*Problema:* Descrições de tabelas e colunas precisam ser aplicadas no Unity Catalog em todas as camadas sem duplicação e sem acoplar configuração ao código de transformação.
*Decisão:* Dataclasses `ColumnMeta` / `TableMeta` (Python puro, sem dependência de PySpark) centralizam descrições e tipo de cast. Um único `TAXI_META` é compartilhado entre Bronze, Silver e Gold — apenas a descrição da tabela varia por camada (constantes `*_TABLE_COMMENT`). A função `_apply_table_metadata()` filtra silenciosamente colunas ausentes na tabela alvo, tornando o contrato reutilizável independente de quantas colunas cada camada expõe.

**4. Silver como Single Source of Truth — Filtro Estrutural vs. Filtro de Negócio**
*Problema:* O case permite cortar colunas e "limpar" os dados. Filtros agressivos baseados em valores atípicos causariam perda de contexto analítico.
*Decisão:*
- Manter todas as 19 colunas: sem `payment_type`, não é possível contextualizar `tip_amount=0` (cash não registra gorjeta) nem `total_amount < 0` (chargeback de disputa).
- Remover apenas impossibilidades físicas: `dropoff < pickup`, datas fora do range Jan–Mai 2023, duplicatas por chave composta.
- Manter `total_amount < 0`: EDA confirmou que a maioria é `payment_type=4` (Dispute) — chargeback é evento de negócio legítimo, não corrupção de dado.
- Manter NULLs sistêmicos (428k linhas, 5 colunas simultâneas): falha de fornecedor em campos secundários; âncora temporal e valor financeiro permanecem válidos.
- Manter `passenger_count=0` e `trip_distance=0`: decisão de negócio para o consumidor, não da plataforma.

**5. Scaffolding de Camadas Legível**
*Decisão:* Pasta `ingestion/` renomeada para `raw/` para que a estrutura de `src/` reflita a progressão do pipeline diretamente: `raw/ → bronze/ → silver/ → gold/`. Nenhuma pasta `etl/` intermediária — `src/` já é o pacote ETL, adicionar um nível seria nesting sem ganho semântico.

**6. Notebook Narrativo como Entrega**
*Decisão:* Em vez de documentação separada, os notebooks seguem uma jornada de Data Storytelling: EDA prova a necessidade → decisão de limpeza é justificada com números → Silver é construída → Gold responde as perguntas de negócio. O avaliador acompanha o raciocínio, não apenas vê o resultado.

**7. Modelagem da Camada Gold — Regras de Negócio vs. Modelagem Dimensional**
*Problema:* Colunas de código numérico (`VendorID`, `RatecodeID`, `payment_type`) são opacas para consumidores analíticos. A alternativa canônica seria tabelas dimensão (Kimball/star schema). A questão é se o investimento se justifica para o escopo do case.
*Decisão:* Para o escopo delimitado do case e o baixo volume de dados, aplicamos enriquecimento via `CASE WHEN` diretamente na Gold — suficiente para tornar a camada de consumo auto-explicativa sem overhead arquitetural. Tabelas dimensão **não** foram implementadas: três dims com 4, 7 e 7 linhas cada não demonstram visão de escala, demonstram aplicação de padrão sem avaliação de contexto. Em um cenário produtivo e visando escala — múltiplos tipos de táxi, múltiplos anos, equipes de analytics distintas consumindo — a modelagem Kimball seria o caminho correto: dims de fornecedor, código de tarifa e método de pagamento em Silver, com fact table em Gold. `PULocationID`/`DOLocationID` mantidos como IDs numéricos: o arquivo de lookup de zonas TLC não faz parte do dicionário recebido e não é exigido pelas perguntas do case.